<a href="https://colab.research.google.com/github/Argus-a/proj2/blob/all-models/%D8%A8%D8%A7%D8%B1%D8%AA_%D9%88_%D8%B1%D8%A7_%D9%86%D8%AF%D9%88%D9%85_%D9%88_%D9%83%D9%88%D9%8A%D9%86_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = '/content/drive/MyDrive/robert 2-5'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("Successfully loaded the model and tokenizer for classification!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Successfully loaded the model and tokenizer for classification!


In [3]:

num_labels = model.config.num_labels
print(f"النموذج مجهز لتصنيف عدد من المشاعر يساوي: {num_labels}")

if hasattr(model.config, 'id2label') and len(model.config.id2label) > 0:
    print("التصنيفات المتاحة في النموذج:")
    for key, value in model.config.id2label.items():
        print(f"{key}: {value}")

النموذج مجهز لتصنيف عدد من المشاعر يساوي: 6
التصنيفات المتاحة في النموذج:
0: sadness
1: joy
2: love
3: anger
4: fear
5: surprise


In [4]:
import pandas as pd
sleep_efficiency_df = pd.read_csv('/content/drive/MyDrive/Sleep_Efficiency_Project/cleaned_sleep_data.csv')
print("First 5 rows of 'cleaned_sleep_data':")
display(sleep_efficiency_df.head())

First 5 rows of 'cleaned_sleep_data':


,ID,Age,Gender,Bedtime,Wakeup time,Sleep duration,Sleep efficiency,REM sleep percentage,Deep sleep percentage,Light sleep percentage,Awakenings,Caffeine consumption,Alcohol consumption,Smoking status,Exercise frequency
0,1,65,Female,2021-03-06 01:00:00,2021-03-06 07:00:00,6.0,0.88,18,70,12,0.0,0.0,0.0,Yes,3.0
1,2,69,Male,2021-12-05 02:00:00,2021-12-05 09:00:00,7.0,0.66,19,28,53,3.0,0.0,3.0,Yes,3.0
2,3,40,Female,2021-05-25 21:30:00,2021-05-25 05:30:00,8.0,0.89,20,70,10,1.0,0.0,0.0,No,3.0
3,4,40,Female,2021-11-03 02:30:00,2021-11-03 08:30:00,6.0,0.51,23,25,52,3.0,50.0,5.0,Yes,1.0
4,5,57,Male,2021-03-13 01:00:00,2021-03-13 09:00:00,8.0,0.76,27,55,18,3.0,0.0,3.0,No,3.0


In [5]:
import joblib
rf_model_path = '/content/drive/MyDrive/Sleep_Efficiency_Project/random_forest_sleep_efficiency_model.joblib'
rf_model = joblib.load(rf_model_path)

print("Successfully loaded the Random Forest model!")
display(rf_model)

Successfully loaded the Random Forest model!


RandomForestRegressor(random_state=42)

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
qwen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("تم تحميل Qwen")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

تم تحميل Qwen


In [7]:
def analyze_and_respond(user_text, sleep_features):
    """
    دالة متكاملة لربط النماذج الثلاثة معاً
    user_text: النص الذي أدخله المستخدم (لتحديد الشعور)
    sleep_features: بيانات النوم بصيغة DataFrame أو 2D array لنموذج Random Forest
    """
    # 1. تحديد الشعور عبر نموذج RoBERTa للتصنيف
    inputs = tokenizer(user_text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)

    # استخراج الشعور الحقيقي من المخرجات (logits)
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()
    sentiment_label = model.config.id2label[predicted_class_id]

    emotion_translation = {
        "sadness": "حزين/مكتئب",
        "joy": "سعيد/فرحان",
        "love": "محب/عاطفي",
        "anger": "غاضب/منفعل",
        "fear": "خائف/قلق",
        "surprise": "متفاجئ/مندهش"
    }
    sentiment = emotion_translation.get(sentiment_label, sentiment_label)

    # 2. التنبؤ بجودة النوم عبر نموذج Random Forest
    sleep_quality_pred = rf_model.predict(sleep_features)[0]

    # 3. تمرير النتائج إلى نموذج Qwen لتوليد الرد
    prompt = f"المستخدم يشعر حالياً بـ ({sentiment}) بناءً على تحليلات كلامه. وبناءً على بيانات نومه، جودة نومه المتوقعة هي: ({sleep_quality_pred}). قدم له رداً مناسباً ونصيحة مختصرة ولطيفة باللغة العربية."

    messages = [
        {"role": "system", "content": "أنت مساعد ذكي وطبيب مختص في جودة النوم والصحة النفسية."},
        {"role": "user", "content": prompt}
    ]

    text = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = qwen_tokenizer([text], return_tensors="pt").to(qwen_model.device)

    # توليد النص
    generated_ids = qwen_model.generate(
        model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=256,
        temperature=0.7
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = qwen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return sentiment, sleep_quality_pred, response

print("تم تحديث الدالة لتصنيف المشاعر الحقيقية بنجاح!")

تم تحديث الدالة لتصنيف المشاعر الحقيقية بنجاح!


### تجربة الدالة الشاملة
يمكنك الآن استدعاء الدالة من خلال إعطائها نص يعبر عن شعور المستخدم، وصف واحد (أو عينة واحدة) من بيانات النوم التي يطلبها نموذج `Random Forest`.

In [8]:
user_input = "i am not good i didn sleep well."

columns_to_drop = ['Sleep efficiency', 'Bedtime', 'Gender', 'ID', 'Smoking status', 'Wakeup time']


sample_sleep_features = sleep_efficiency_df.sample(1).copy()
for col in columns_to_drop:
    if col in sample_sleep_features.columns:
        sample_sleep_features = sample_sleep_features.drop(columns=[col])

try:

    sentiment, sleep_quality, final_response = analyze_and_respond(user_input, sample_sleep_features)

    print("--- تحليل الشعور (RoBERTa) ---")
    print(sentiment)

    print("\n--- جودة النوم المتوقعة (Random Forest) ---")
    print(sleep_quality)

    print("\n--- رد المساعد (Qwen) ---\n")
    print(final_response)
except Exception as e:
    print(f"حدث خطأ أثناء التنفيذ، قد تحتاج لتعديل ميزات البيانات الممررة لنموذج Random Forest: {e}")

--- تحليل الشعور (RoBERTa) ---
خائف/قلق

--- جودة النوم المتوقعة (Random Forest) ---
0.6009000000000007

--- رد المساعد (Qwen) ---

أنا أفهم أنك تعاني من القلق أو الخوف الحاد. هذه المشاعر قد تكون مؤذية لكنها عادة ما تمر بسرعة. يمكنك الاستماع إلى الموسيقى الهادئة أو القراءة لمساعدتك في الهدوء. لا تتردد في التحدث مع شخص آمن عن مشاعرك إذا كنت ترغب في ذلك. في حالة عدم تحسن الأمور بعد أسبوعين، فقد يكون من الأفضل زيارة الطبيب. 

بالنسبة للنوم، يبدو أنك تواجه صعوبة في الحصول على النوم الكافي. قد تحتاج إلى تنظيم حياتك اليومية أو اتباع نظام غذائي صحي. إذا استمرت مشاكل النوم، فلا تتردد في الرجوع إلى الطبيب. 

تذكر دائمًا أن الصحة النفسية والجسدية هما جزءان من بعضهما البعض. استخدم كل الموارد المتاحة لك لتعزيز سلوك النوم الجيد والشعور بالراحة.


### تجربة الدالة الشاملة بنص مخصص من اختيارك
يمكنك كتابة أي نص في الخلية التالية لتجربة التحليل بشكل تفاعلي.

In [13]:
#@title اكتب نصك هنا لتجربته مع النموذج

user_custom_input = ".تأتي لي افكار مزعجه وتخرب مزاجي, لا استطيع النوم جيدا منها" #@param {type:"string"}

# الأعمدة التي لم يتعرف عليها النموذج
columns_to_drop = ['Sleep efficiency', 'Bedtime', 'Gender', 'ID', 'Smoking status', 'Wakeup time']

# اختيار صف عشوائي لتجربة ميزات النوم
sample_sleep_features = sleep_efficiency_df.sample(1).copy()
for col in columns_to_drop:
    if col in sample_sleep_features.columns:
        sample_sleep_features = sample_sleep_features.drop(columns=[col])

try:
    # استدعاء الدالة
    sentiment, sleep_quality, final_response = analyze_and_respond(user_custom_input, sample_sleep_features)

    print("--- تحليل الشعور (RoBERTa) ---")
    print(sentiment)

    print("\n--- جودة النوم المتوقعة (Random Forest) ---")
    print(f"{sleep_quality:.2%}")

    print("\n--- رد المساعد (Qwen) ---\n")
    print(final_response)
except Exception as e:
    print(f"حدث خطأ أثناء التنفيذ: {e}")

--- تحليل الشعور (RoBERTa) ---
غاضب/منفعل

--- جودة النوم المتوقعة (Random Forest) ---
54.51%

--- رد المساعد (Qwen) ---

عذرًا على أي استفزاز قد أصابك. الظروف التي تمر بها الآن قد تكون صعبة، لكن لا تقلق؛ هناك طرق للتعامل معها بشكل سلبي يمكن أن تؤدي إلى ضعف النوم. احرص على تنظيم حياتك اليومية وتجنب التوتر الجسيم. إذا كنت تعاني من الغضب المستمر، فقد يكون من المفيد البحث عن المساعدة أو التحدث مع شخص موثوق به في حالة الحاجة. حاول الاسترخاء والاستمتاع بالأمور الإيجابية في حياتك، وأتمنى لك أفضل النوم.


In [10]:
# import os
# import joblib

# # تحديد المسار الرئيسي لحفظ النماذج في Google Drive
# drive_save_dir = '/content/drive/MyDrive/proj2_models'
# os.makedirs(drive_save_dir, exist_ok=True)

# print("جاري حفظ النماذج في Google Drive...")

# # 1. حفظ نموذج RoBERTa والمحلل
# roberta_path = os.path.join(drive_save_dir, 'roberta_sentiment_model')
# model.save_pretrained(roberta_path)
# tokenizer.save_pretrained(roberta_path)
# print(f"- تم حفظ نموذج RoBERTa بنجاح في: {roberta_path}")

# # 2. حفظ نموذج Random Forest
# rf_path = os.path.join(drive_save_dir, 'random_forest_model.joblib')
# joblib.dump(rf_model, rf_path)
# print(f"- تم حفظ نموذج Random Forest بنجاح في: {rf_path}")

# # 3. حفظ نموذج Qwen والمحلل
# qwen_path = os.path.join(drive_save_dir, 'qwen_assistant_model')
# qwen_model.save_pretrained(qwen_path)
# qwen_tokenizer.save_pretrained(qwen_path)
# print(f"- تم حفظ نموذج Qwen بنجاح في: {qwen_path}")

# print("\nاكتمل حفظ جميع النماذج بنجاح في Google Drive!")